# Generacion de Texto con LSTM

Modelo de lenguaje: modelo que puede predecir la probabilidad del proximo token dado el anterior.

Flujo:
1. Preparar los **datos de acondicionamiento** (cadena inicial de texto).
2. Generar siguiente token.
3. Añadir token generado a los datos de entrada.
4. Repetir desde el paso 2.

![Diagrama de generacion de tokens a nivel de caracteres de forma iterativa. Encadenando la salida de cada iteracion con la entrada para generar la entrada de la siguiente iteración.](./images/8.1.1.lenguage-model-character-level.png)


Sobre la distribución que genera el modelo de lenguaje hay que elegir el siguiente caracter. A esto se llama *Sampling strategy* (estrategia de muestreo). 

Una estrategia ingenua es siempre elegir el caracter mas probable (*Greedy Sampling*). Pero termina generando cadenas repetitivas y predecibles que no necesariamente se corresponden a un lenguaje coherente.

Una aproximacion mas interesante, es introducir de aleatoriedad proporcional a la distribucion de probabilidad. lo que se llama  *Stochastic Sampling*. Por ejemplo si *e* tiene probabilidad $0.3$ entonces sera escogido el $30\%$ de las veces.

Hay un problema con esa estrategia. No ofrece una forma de controlar la cantidad de aleatoriedad Se quiere un poco para generar secuencias creativas y sorpresivas. Pero no demasiada ya que las secuencias se vuelven incoherentes.

Bajo el orden de **regular la cantidad de entropia** se introduce el parametro ***softmax temperature***. El mismo regula la aleatoridad las distribuciones de probabilidad que luego se usan en la eleccion del proximo token.

Dada un valor de *Temperatura*, se computa una nueva distribución de probabilidad desde la original. A continuacion se muestra un ejemplo de como se realizar esta reponderación.

In [10]:
import numpy as np

def reweight_distribution(original_distribution, temperature=0.5):
    """
    La distribucion original es un arreglo 1D de probabilidades que suman 1. 
    La termperatura cuantifica la entropia de la distribucion.

    Retorna la version reponderada. La suma de la distribucion reponderada es 1.
    """
    distribution = np.log(original_distribution) / temperature
    print(f"Distribution log/temperature: {distribution}")
    distribution = np.exp(distribution)
    print(f"Distribution exp: {distribution}")

    return distribution / np.sum(distribution)

# Ejemplo de uso
distribution = np.array([0.1, 0.2, 0.3, 0.4])
print(f"Original distribution: {distribution}")
rewighted_distribution = reweight_distribution(distribution, temperature=0.5)
print(f"Reweighted distribution: {rewighted_distribution}")

Original distribution: [0.1 0.2 0.3 0.4]
Distribution log/temperature: [-4.60517019 -3.21887582 -2.40794561 -1.83258146]
Distribution exp: [0.01 0.04 0.09 0.16]
Reweighted distribution: [0.03333333 0.13333333 0.3        0.53333333]


A continuacion se muestra un grafico ilustrativo de como las temperaturas bajas convergen la probabilidades en el caracter mas probable. Temperaturas mas altas equiparan todas las probabilidades en un mismo nivel. Mantener la temperatura en 1 hace que la distribucion se mantenga igual.

![Grafico ilustrativo de como las temperaturas bajas convergen la probabilidades en el caracter mas probable. Llevar la temperatura en 1 hace que la distribucion se mantenga igual.](./images/8.1.2.distribution-graph-with-many-temperatures.png)

# Implementacion de LSTM de generación a nivel de caracter

En este ejemplo se usan algunos de los escritos de Nietzsche (filoso de finales del siglo 19). Los textos estan ingles por lo que el modelo entrenado respondera a ese idioma.

El modelo respetara el estilo de escritura de Nietzche y los topicos que manejaba.

## Corpus

A continuacion se muestra como descarcargar los datos.

In [2]:
import keras
import numpy as np

path = keras.utils.get_file(
    'nietzsche.txt',
    origin='https://s3.amazonaws.com/text-datasets/nietzsche.txt',
    cache_dir=".",
    cache_subdir="datasets")
text = open(path, encoding='utf-8').read().lower()
print(f'Corpus length: {len(text)}')

2026-07-23 20:54:36.447898: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-07-23 20:54:36.447934: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-07-23 20:54:36.449256: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-07-23 20:54:36.456081: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Corpus length: 600893


Hay que estandarizar los datos de entrenamiento antes de entrenar el modelo. 

Para ello se van a extraer secuencias parcialmente superpuestas de un largo acotado. Este sera el arreglo `x`.

Tambien se prepara una entrada en un arreglo `y` con los objetivos a predecir por cada `x`.

Posteriormente se aplicara una codificacion *one-hot* a nivel de caracter.

In [ ]:
maxlen = 60
step = 3        # Nueva secuencia cada 3 caracteres
sentences = []  # Secuencias extraidas
next_chars = []  # Caracteres siguientes a las secuencias extraidas

for i in range(0, len(text) - maxlen, step):
    sentences.append(text[i: i + maxlen])
    next_chars.append(text[i+maxlen])
    # print(f"> {text[i: i + maxlen]} -> {text[i+maxlen]}")  # Comprobar pares de entrenamiento generados
print(f'Number of sequences: {len(sentences)}')

chars = sorted(list(set(text)))                                     # Lista de caracteres unicos
print(f'Unique characters: {len(chars)}')
char_indices = dict((char, chars.index(char)) for char in chars)    # Diccionario caracter -> indice

print('Vectorización...')
x = np.zeros(
    (len(sentences), maxlen, len(chars)), 
    dtype=bool)
y = np.zeros(
    (len(sentences), len(chars)),
    dtype=bool)
for i, sentence in enumerate(sentences):
    for t, char in enumerate(sentence):
        x[i, t, char_indices[char]] = 1
    y[i, char_indices[next_chars[i]]] = 1
#print(f'x: {x}')
#print(f'y: {y}')

Number of sequences: 200278
Unique characters: 57
Vectorización...
x: [[[False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]

 [[False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False  True False ... False False False]]

 [[False False False ... False False False]
  [ True False False ... False False False]
  [ True False False ... False False False]
  ...
  [False False False ... False False False]
  [False False False ... False False False]
  [False False False ... False False False]]

 ...

 [[False False False ... False False False]
  [False  True False ... False False False]
  [False False False

Al tomar secuencias cada 3 caracteres se generaron aproximadamente 1/3 de entradas en relacion al largo total del texto. La cantidad de muestras respeta la siguiente formula:
$$
\text{len(secuencias)} = \left\lfloor \frac{\text{len(text)} - \text{maxlen}}{\text{step}} \right\rfloor
$$